# The Spring Element: Designing the Elastic Foundation
### *Modern Strain Gage Technology* — Companion Notebook

**Every strain gage transducer is only as good as the piece of metal it is bonded to.** The *spring element* is that piece of metal — the mechanical component that deforms predictably under the input quantity (force, pressure, torque) and presents a well-defined strain field to the gages on its surface. It must be stiff enough not to disturb the system being measured (a force transducer should not deflect the structure it reads), yet compliant enough to produce measurable strain at the expected load.

The three classical spring element geometries serve different measurement needs. A *column* in axial compression is extremely stiff but produces low strain per unit force — suited to very large loads. A *cantilever beam* is far more compliant and produces high bending strain per unit force — suited to precision force measurement from newtons to kilonewtons. A *shear-web* beam exploits the shear stress distribution to minimize the influence of bending moments at the mounting points. Understanding the stiffness and strain sensitivity of each geometry is the first step in transducer design.

Companion notebook to Chapter 40 — The Spring Element: Designing the Elastic Foundation.

**This notebook demonstrates:**
1. Stiffness (N/mm) and strain sensitivity (με/N) for the three fundamental spring element geometries — column, cantilever beam, and shear-web — using the strain relations of §40.2 and §40.4.
2. The two schematic figures used in the chapter (Figures 40.1 and 40.2), generated here so they stay fully reproducible.

In [ ]:
# !pip install numpy matplotlib
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon, Circle, Ellipse
from matplotlib.transforms import Affine2D

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

---
## 1 — Spring Element Stiffness and Strain Sensitivity

The most striking result below is the enormous sensitivity difference between the column and the cantilever beam. A steel column with a 100 mm² cross-section produces about 0.05 με/N of axial strain — a very low sensitivity that requires large forces (tens of kilonewtons) to produce measurable bridge output. The same material in a cantilever geometry produces 16.7 με/N of bending strain at the root — roughly 333× more sensitive per newton applied. The tradeoff is stiffness: this cantilever is about 3,700× less stiff than the column.

The shear-web falls between the two in sensitivity (0.13 με/N) and offers a different advantage — its output strongly *rejects* bending moments applied at the mounting points (good rejection, not perfect immunity — off-axis sensitivity still depends on machining tolerance and bridge balance, as §40.3 stresses), making it useful for force transducers where the load path is not perfectly controlled. One subtlety: the shear value below is the engineering shear strain *γ* at the web's neutral axis, and the ±45° gages actually read the normal strain *ε = γ/2* along the principal directions — so their per-newton strain reading is half the quoted γ. Either way, the shear-web sits between the column and the cantilever. These stiffness and sensitivity numbers are the starting inputs to any transducer design; with them in hand, material and geometry can be iterated to meet the target specifications.

In [ ]:
# --- Material: structural steel -------------------------------------------
E_STEEL = 200e3                              # elastic modulus, MPa (= 200 GPa)
NU_STEEL = 0.30                              # Poisson's ratio, dimensionless
G_STEEL = E_STEEL / (2 * (1 + NU_STEEL))     # shear modulus, MPa
UE_PER_STRAIN = 1e6                          # microstrain per unit strain


def column_response(area, length, E=E_STEEL):
    """Column in axial load.

    Returns (k, sensitivity):
      k           axial stiffness EA/L, in N/mm
      sensitivity axial strain per unit force 1/(A*E), in me/N (microstrain/N)
    With E in MPa (N/mm^2) and area in mm^2, E*A/L has units N/mm.
    """
    k = E * area / length
    sensitivity = 1.0 / (area * E) * UE_PER_STRAIN
    return k, sensitivity


def cantilever_response(width, depth, length, E=E_STEEL):
    """Rectangular cantilever loaded transversely at the free end.

    Returns (k, sensitivity):
      k           tip stiffness 3EI/L^3, in N/mm  (I = b*h^3/12)
      sensitivity root surface strain per unit force 6L/(E*b*h^2), in me/N
    Implements the chapter's Eq. (40.1), eps_max = 6FL/(E*b*h^2), at F = 1 N.
    """
    I = width * depth**3 / 12.0
    k = 3 * E * I / length**3
    sensitivity = 6 * length / (width * depth**2 * E) * UE_PER_STRAIN
    return k, sensitivity


def shear_web_response(area, G=G_STEEL):
    """Shear-web element at the neutral axis (rectangular section).

    Returns the engineering shear-strain sensitivity gamma per unit force,
    3/(2*A*G), in me/N. Peak shear stress tau = 3V/(2A); gamma = tau/G.
    The +/-45 gages read normal strain eps = gamma/2 along the principal axes.
    """
    return 3.0 / (2 * area * G) * UE_PER_STRAIN


# --- Representative geometries (mm) ---------------------------------------
COLUMN_AREA, COLUMN_LENGTH = 100.0, 50.0        # mm^2, mm
BEAM_WIDTH, BEAM_DEPTH, BEAM_LENGTH = 10.0, 3.0, 50.0   # mm
SHEAR_WEB_AREA = 150.0                           # mm^2

k_col, sens_col = column_response(COLUMN_AREA, COLUMN_LENGTH)
k_beam, sens_beam = cantilever_response(BEAM_WIDTH, BEAM_DEPTH, BEAM_LENGTH)
gamma_shear = shear_web_response(SHEAR_WEB_AREA)

print("SPRING ELEMENT STIFFNESS AND STRAIN SENSITIVITY (steel, E = 200 GPa)")
print("=" * 68)
print(f"1. Column (axial load), A = {COLUMN_AREA:.0f} mm^2, L = {COLUMN_LENGTH:.0f} mm")
print(f"   k = {k_col:>10.0f} N/mm    sensitivity = {sens_col:7.4f} me/N (axial strain)")
print(f"2. Cantilever (bending), b = {BEAM_WIDTH:.0f}, h = {BEAM_DEPTH:.0f}, L = {BEAM_LENGTH:.0f} mm")
print(f"   k = {k_beam:>10.2f} N/mm    sensitivity = {sens_beam:7.4f} me/N (root surface strain)")
print(f"3. Shear-web, A = {SHEAR_WEB_AREA:.0f} mm^2, G = {G_STEEL:.0f} MPa")
print(f"   gamma sensitivity = {gamma_shear:7.4f} me/N   (+/-45 gages read eps = gamma/2 = {gamma_shear/2:.4f} me/N)")

---
## 2 — Spring Element Diagrams (Figures 40.1 and 40.2)

The two schematic figures used in the chapter are generated here so they stay reproducible. **Figure 40.1** shows the *bending* family as a progression — basic cantilever, width-tapered constant-stress beam, coupled dual-beam (binocular), and ring — matching the discussion in §40.2. **Figure 40.2** lines up representative elements from all three fundamental categories — a cantilever and a binocular (both bending), a shear-web, and a column — that anchor the selection guidance in §40.5. Both write PNGs into `src/media/ch40/`.

In [ ]:
# ---- palette (matches the book's other spring-element art) ----------------
BODY, EDGE = "#d7e6f7", "#1f3a68"
GAGE, GAGE_E = "#f4c542", "#8a6d1a"
GAGER, GAGEB = "#c0392b", "#1f6fc0"
LOAD, WALL, TXT = "#1f3a68", "#6b6b6b", "#333333"

# resolve src/media/ch40 whether run from notebooks/ or the repo root
_root = os.getcwd()
if os.path.basename(_root) == "notebooks":
    _root = os.path.dirname(_root)
MEDIA = os.path.join(_root, "src", "media", "ch40")
os.makedirs(MEDIA, exist_ok=True)


def wall(ax, x, y0, y1, w=0.18):
    """Draw a hatched fixed-support (ground) wall of width w at x, spanning y0..y1."""
    ax.add_patch(Rectangle((x - w, y0), w, y1 - y0, facecolor=WALL,
                           edgecolor="none", zorder=3))
    for yy in np.linspace(y0, y1, 9):
        ax.plot([x - w, x - w - 0.12], [yy, yy - 0.10], color=WALL, lw=1.1, zorder=3)


def gage(ax, cx, cy, w=0.30, h=0.16, color=GAGE, ec=GAGE_E, angle=0):
    """Draw a strain gage as a filled rectangle centered at (cx, cy), optionally rotated."""
    r = Rectangle((cx - w/2, cy - h/2), w, h, facecolor=color, edgecolor=ec, lw=1.2, zorder=6)
    if angle:
        r.set_transform(Affine2D().rotate_deg_around(cx, cy, angle) + ax.transData)
    ax.add_patch(r)


def load_arrow(ax, x, y, dy, label="F", lx=0.0):
    """Draw a load arrow of vertical length dy from (x, y) and annotate it with label."""
    ax.annotate("", xy=(x, y + dy), xytext=(x, y),
                arrowprops=dict(arrowstyle="-|>", color=LOAD, lw=2.4))
    if label:
        ax.text(x + lx, y + dy + (0.12 if dy > 0 else -0.18), label, ha="center",
                va="bottom" if dy > 0 else "top", fontstyle="italic", fontsize=13, color=LOAD)

# ============ Figure 40.1 — bending-element progression ====================
fig, axes = plt.subplots(1, 4, figsize=(15, 4.4))
for ax in axes:
    ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-0.6, 4.2); ax.set_ylim(-1.6, 2.4)

ax = axes[0]
ax.set_title("Basic Cantilever", fontsize=12, fontstyle="italic", color=EDGE, pad=6)
wall(ax, 0.0, -0.5, 0.5)
ax.add_patch(Rectangle((0.0, -0.35), 3.2, 0.7, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
gage(ax, 0.55, 0.35, w=0.34); gage(ax, 0.55, -0.35, w=0.34)
load_arrow(ax, 3.1, -0.35, -0.9, "F")
ax.text(0.55, 0.78, "gages", ha="center", fontsize=9.5, color=GAGE_E)
ax.text(1.7, -1.15, "strain peaks at the root,\nfalls off toward the tip", ha="center", fontsize=9.5, color=TXT)

ax = axes[1]
ax.set_title("Constant-Stress Taper", fontsize=12, fontstyle="italic", color=EDGE, pad=6)
wall(ax, 0.0, -0.7, 0.7)
ax.add_patch(Polygon([(0.0, 0.7), (0.0, -0.7), (3.2, -0.06), (3.2, 0.06)], closed=True,
                     facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
for gx in (0.7, 1.5, 2.3):
    gage(ax, gx, 0.0, w=0.30, h=0.14)
load_arrow(ax, 3.15, 0.0, -0.9, "F")
ax.text(1.6, 1.05, "(plan view)", ha="center", fontsize=9, color=TXT, fontstyle="italic")
ax.text(1.6, -1.15, "width narrows to hold\nuniform strain along the span", ha="center", fontsize=9.5, color=TXT)

ax = axes[2]
ax.set_title("Coupled Dual-Beam (Binocular)", fontsize=12, fontstyle="italic", color=EDGE, pad=6)
ax.add_patch(Rectangle((0.0, -0.95), 3.2, 1.9, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
ax.add_patch(Ellipse((1.6, 0.0), 1.7, 0.55, facecolor="white", edgecolor=EDGE, lw=1.4, zorder=4))
for gx in (1.15, 2.05):
    gage(ax, gx, 0.62, w=0.34); gage(ax, gx, -0.62, w=0.34)
ax.add_patch(Rectangle((0.0, -0.95), 0.45, 1.9, facecolor="#c2d4ea", edgecolor=EDGE, lw=1.2, zorder=3))
ax.add_patch(Rectangle((2.75, -0.95), 0.45, 1.9, facecolor="#c2d4ea", edgecolor=EDGE, lw=1.2, zorder=3))
load_arrow(ax, 3.0, 0.95, 0.85, "F")
ax.text(1.6, -1.5, "sensing-beam moments nearly\nindependent of load position", ha="center", fontsize=9.5, color=TXT)

ax = axes[3]
ax.set_title("Ring Element", fontsize=12, fontstyle="italic", color=EDGE, pad=6)
cx, cy, ro, ri = 1.7, 0.0, 1.05, 0.72
ax.add_patch(Circle((cx, cy), ro, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
ax.add_patch(Circle((cx, cy), ri, facecolor="white", edgecolor=EDGE, lw=1.6, zorder=3))
gage(ax, cx - (ro + ri)/2, cy, w=0.22, h=0.30); gage(ax, cx + (ro + ri)/2, cy, w=0.22, h=0.30)
load_arrow(ax, cx, cy + ro, 0.7, "F"); load_arrow(ax, cx, cy - ro, -0.7, "")
ax.text(cx, -1.5, "bending + direct stress\nin a closed geometry", ha="center", fontsize=9.5, color=TXT)

fig.suptitle("Bending Spring Element Progression", fontsize=14, fontweight="bold", color=EDGE, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(MEDIA, "fig40_1_bending_progression.png"), dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

# ============ Figure 40.2 — three fundamental types compared ===============
fig, axes = plt.subplots(1, 4, figsize=(15, 5.2))
for ax in axes:
    ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-0.8, 3.4); ax.set_ylim(-2.4, 3.0)

ax = axes[0]
ax.set_title("Basic\nCantilever", fontsize=12, fontweight="bold", color=EDGE, pad=4)
wall(ax, 0.0, -0.45, 0.45)
ax.add_patch(Rectangle((0.0, -0.32), 2.6, 0.64, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
gage(ax, 0.5, 0.32, w=0.34)
load_arrow(ax, 2.5, -0.32, -0.85, "F")
ax.text(1.3, -1.5, "max strain at root;\nprecise gage placement", ha="center", fontsize=9.5, color=TXT)

ax = axes[1]
ax.set_title("Binocular\nDual-Beam", fontsize=12, fontweight="bold", color=EDGE, pad=4)
ax.add_patch(Rectangle((0.0, -0.85), 2.6, 1.7, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
ax.add_patch(Ellipse((1.3, 0.0), 1.5, 0.5, facecolor="white", edgecolor=EDGE, lw=1.4, zorder=4))
ax.add_patch(Rectangle((0.0, -0.85), 0.4, 1.7, facecolor="#c2d4ea", edgecolor=EDGE, lw=1.2, zorder=3))
ax.add_patch(Rectangle((2.2, -0.85), 0.4, 1.7, facecolor="#c2d4ea", edgecolor=EDGE, lw=1.2, zorder=3))
for gx in (0.95, 1.65):
    gage(ax, gx, 0.55, w=0.30); gage(ax, gx, -0.55, w=0.30)
load_arrow(ax, 2.4, 0.85, 0.75, "F")
ax.text(1.3, -1.5, "uniform strain, double output;\nlow load-position sensitivity", ha="center", fontsize=9.5, color=TXT)

ax = axes[2]
ax.set_title("Shear-Web", fontsize=12, fontweight="bold", color=EDGE, pad=4)
ax.add_patch(Rectangle((0.2, 0.6), 2.2, 0.55, facecolor="#c2d4ea", edgecolor=EDGE, lw=1.4, zorder=2))
ax.add_patch(Rectangle((0.2, -1.15), 2.2, 0.55, facecolor="#c2d4ea", edgecolor=EDGE, lw=1.4, zorder=2))
ax.add_patch(Rectangle((1.05, -0.6), 0.5, 1.2, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
gage(ax, 1.3, 0.0, w=0.5, h=0.13, color=GAGEB, ec=EDGE, angle=45)
gage(ax, 1.3, 0.0, w=0.5, h=0.13, color=GAGER, ec=EDGE, angle=-45)
load_arrow(ax, 1.3, 1.15, 0.7, "V")
ax.text(1.3, -1.9, "gages at ±45° in pure shear;\ntolerant of load position", ha="center", fontsize=9.5, color=TXT)

ax = axes[3]
ax.set_title("Column\n(Compression)", fontsize=12, fontweight="bold", color=EDGE, pad=4)
ax.add_patch(Rectangle((0.7, -1.15), 1.2, 2.3, facecolor=BODY, edgecolor=EDGE, lw=1.6, zorder=2))
gage(ax, 0.7, 0.1, w=0.14, h=0.4); gage(ax, 1.9, 0.1, w=0.14, h=0.4)
load_arrow(ax, 1.3, 1.15, 0.7, "F")
ax.annotate("", xy=(1.3, -1.85), xytext=(1.3, -1.15), arrowprops=dict(arrowstyle="-|>", color=LOAD, lw=2.4))
ax.text(1.3, -2.15, "uniform axial strain;\ncompact at high load", ha="center", fontsize=9.5, color=TXT)

fig.suptitle("Spring Element Types Compared", fontsize=14, fontweight="bold", color=EDGE, y=1.0)
fig.text(0.5, -0.02, "Capacity ranges are indicative; actual values depend on geometry, material, and accuracy class.",
         ha="center", fontsize=9, fontstyle="italic", color=TXT)
fig.tight_layout()
fig.savefig(os.path.join(MEDIA, "fig40_2_spring_comparison.png"), dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print("Figures written to", MEDIA)

---
## Where to take this next

The comparison above is deliberately minimal — a single operating point per geometry. A few concrete extensions turn it into a real design tool:

1. **Solve the inverse problem, as the chapter's worked example does.** Fix a target strain (say 1,500 με at rated load) and back out the beam length `L` from Eq. (40.1), then sweep `E`, `b`, and `h` to see which geometries can hit that target while keeping peak stress below the material's fatigue limit. Reproduce the chapter's 50‑lb, 2024‑T81 aluminum cantilever (answer: L ≈ 1.66 in) as a check.
2. **Add the bridge and the estimated first natural frequency.** Convert each element's strain sensitivity into bridge output using the full‑bridge relations (GF·ε for a fully active bending bridge; the (1+ν)/2 factor of Eq. (40.4) for the column), and estimate `f_n ∝ √(k/m)` for a lumped end mass — making explicit the stiffness-vs-sensitivity tradeoff the chapter describes.
3. **Sweep the selection map.** Plot stiffness and sensitivity versus rated capacity for all three geometries on one figure and mark the ~1,000 lb (4.4 kN) bending-to-shear and ~50 kN shear-to-column crossover regions from §40.5, so the "choosing among the three" guidance becomes something you can see rather than memorize.